In [12]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
#Datasets & DataLoader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset=CIFAR10(root='./Data',train=True,download=False,transform=transform)
testset=CIFAR10(root='./Data',train=False,download=False,transform=transform)

In [3]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./Data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [4]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)

# Build CNN

In [5]:
import jupyterlab
print(jupyterlab.__version__)

4.5.9


In [8]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers =nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel size =2 , stride=2

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), 

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) ,
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )
    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1) # falttening
        x=self.fc_layers(x)

        return x

In [9]:
model=CNN()

In [13]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

# Training

In [15]:
epochs=10

for epoch in range(epochs):
    epoch_training_loss=0.0

    for images,labels in trainloader:
        optimizer.zero_grad()
        output =model.forward(images)
        loss=criterion(output,labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss +=loss.item()
    print(f'Epoch={epoch} Loss =>{epoch_training_loss/len(trainloader)}')

Epoch=0 Loss =>1.382648061837077
Epoch=1 Loss =>0.9504150527974834
Epoch=2 Loss =>0.7632082930916105
Epoch=3 Loss =>0.6435729541513316
Epoch=4 Loss =>0.5417892167254177
Epoch=5 Loss =>0.4546937460599043
Epoch=6 Loss =>0.3762531938302852
Epoch=7 Loss =>0.3021158142029629
Epoch=8 Loss =>0.23904086222581547
Epoch=9 Loss =>0.19281217576626242


# Evaluate

In [18]:
correct_labels=0
total_labels=0

model.eval()
with torch.no_grad():
    for images,labels in testloader:
        output=model.forward(images)
        _,predicted=torch.max(output,1)

        correct_labels+=(predicted==labels).sum().item()
        total_labels+=labels.size(0)
print(f'Accuracy = {correct_labels/total_labels*100}%')

Accuracy = 74.76%
